# UI Canvas Experimentation
This notebook implements a specialized `AnnotationCanvas` that follows the logic of `input_manager.js` for handling complex UI events (clicks, drags, double-clicks, keyboard modifiers) in a single-window or multi-window setup.

In [ ]:
import ipywidgets as widgets
from ipyevents import Event
import numpy as np
import time
from PIL import Image
import io
from typing import List, Optional, Tuple, Callable
from dicom_utils import UICanvas, WindowMeta, SimpleRGBWidget

### Wymagania dla `base_widget` (self.w)

Klasa `UICanvas` jest zaprojektowana tak, aby być odseparowana od logiki renderowania. Jednakże, aby mogła ona poprawnie przechwytywać zdarzenia i wyświetlać obraz, przekazywany obiekt `base_widget` (argument `base_widget` w konstruktorze) musi implementować następujący interfejs:

1. **`im_w`**: Atrybut będący instancją `ipywidgets.Image`. Jest to fizyczny element DOM, na którym `ipyevents` ustawia listenery.
2. **`widget`**: Atrybut będący kontenerem (np. `widgets.Box` lub `widgets.VBox`), który zawiera `im_w` oraz ewentualne dodatkowe elementy UI (suwaki, labele). To ten obiekt jest wyświetlany wewnątrz `UICanvas`.
3. **`_update_image(z, hu_range, **kwargs)`**: Metoda odświeżająca zawartość `im_w`. `UICanvas` wywołuje ją przy zdarzeniach takich jak scroll (wheel), aby wymusić przerysowanie.

#### Przykład Minimalnej Implementacji:
```python
class MinimalWidget:
    def __init__(self):
        # 1. Element graficzny
        self.im_w = widgets.Image(format='png', width=100, height=100)
        # 2. Główny kontener
        self.widget = widgets.Box([self.im_w])
        
    def _update_image(self, z, hu_range, **kwargs):
        # Logika aktualizacji self.im_w.value
        pass
```

### Experimentation with single window `Slice_Axial`

In [ ]:
from dicom_utils import SimpleRGBWidget
import numpy as np

# 1. Setup a simple 512x512 RGB canvas
w, h = 512, 512
rgb_array = np.zeros((h, w, 3), dtype=np.uint8)
rgb_array[:, :, 0] = np.linspace(0, 255, w) # Red gradient

base_w = SimpleRGBWidget(rgb_array=rgb_array)

# 2. Define single window meta
axial_meta = WindowMeta(width=w, height=h, offset_x=0, offset_y=0, name='Slice_Axial')

msg_lst = []
def event_receiver(msg):
    
    msg_lst.append(msg)
    

# 4. Initialize UICanvas
ui_canvas = UICanvas(base_w, axial_meta, event_receiver)
ui_canvas.display()

In [ ]:
set(m['eventType'] for m in msg_lst)

In [ ]:
for m in msg_lst:
    if m['eventType'] in ['key_down', 'key_up']:
        print(m)

## Integracja z Zewnętrznymi Klasami Generującymi Obraz

Aby zachować czystość kodu przy współpracy z dużymi obiektami danych, zaleca się stosowanie wzorca **Controller/App**. Separuje on logikę biznesową od szczegółów technicznych UI.

### Kluczowe komponenty:
1. **Model (Silnik)**: Przechowuje stan (np. punkty, kolory) i renderuje obraz. Nie wie o istnieniu `ipywidgets`.
2. **View/Proxy (Widget)**: `SimpleRGBWidget` - służy tylko do wyświetlania.
3. **Controller (LabApp)**: Klasa, która odbiera zdarzenia z `UICanvas`, wywołuje odpowiednie metody silnika i zleca odświeżenie widoku.

### Przykład z Kontrolerem:

In [ ]:
class MyLargeObject:
    """Silnik danych - czysty Model"""
    def __init__(self, w=512, h=512):
        self.w, self.h = w, h
        self.color = [100, 100, 100]
        self.points = []
        self.lines = []
        self.current_line = None
        self.current_pos = None
        self.drag_start_pos = None
        
    def add_point(self, x, y):
        self.points.append((int(x), int(y)))
        
    def change_color(self, delta):
        self.color = [np.clip(c + delta*10, 0, 255) for c in self.color]

    def start_drag(self, x, y):
        self.drag_start_pos = (int(x), int(y))
        self.current_pos = [float(x), float(y)]
        self.current_line = (self.drag_start_pos, (int(x), int(y)))

    def move_drag(self, dx, dy):
        if self.drag_start_pos is not None:
            # Server-side accumulation using dx, dy to calculate position
            self.current_pos[0] += float(dx)
            self.current_pos[1] += float(dy)
            self.current_line = (self.drag_start_pos, (int(self.current_pos[0]), int(self.current_pos[1])))

    def end_drag(self):
        if self.drag_start_pos is not None:
            self.lines.append((self.drag_start_pos, (int(self.current_pos[0]), int(self.current_pos[1]))))
            self.current_line = None
            self.drag_start_pos = None

    def render(self):
        img = np.zeros((self.h, self.w, 3), dtype=np.uint8)
        img[:] = self.color
        for px, py in self.points:
            if 0 <= px < self.w and 0 <= py < self.h:
                img[py-2:py+2, px-2:px+2] = [255, 255, 30]
                
        def draw_line(p0, p1, color, thickness=2):
            x0, y0 = p0
            x1, y1 = p1
            length = int(np.hypot(x1 - x0, y1 - y0))
            if length == 0:
                return
            x_vals = np.linspace(x0, x1, length).astype(int)
            y_vals = np.linspace(y0, y1, length).astype(int)
            for vx, vy in zip(x_vals, y_vals):
                for tx in range(-thickness//2 + 1, thickness//2 + 1):
                    for ty in range(-thickness//2 + 1, thickness//2 + 1):
                        if 0 <= vx+tx < self.w and 0 <= vy+ty < self.h:
                            img[vy+ty, vx+tx] = color

        for p0, p1 in self.lines:
            draw_line(p0, p1, [255, 0, 0]) # Red for permanent
            
        if self.current_line is not None:
            draw_line(self.current_line[0], self.current_line[1], [0, 255, 0]) # Green for current

        return img

class LabApp:
    """Kontroler - łączy zdarzenia UI z logiką Modelu"""
    def __init__(self, engine, proxy_widget):
        self.engine = engine
        self.proxy = proxy_widget
        
    def handle_ui_event(self, msg):
        # Tłumaczenie zdarzeń UI na akcje Modelu
        action_taken = False
        
        if msg['eventType'] == 'click':
            self.engine.add_point(msg['x'], msg['y'])
            action_taken = True
            
        elif msg['eventType'] == 'mouse_wheel':
            self.engine.change_color(msg['deltaY'])
            action_taken = True
            
        elif msg['eventType'] == 'drag_start':
            self.engine.start_drag(msg['x'], msg['y'])
            action_taken = True
            
        elif msg['eventType'] == 'drag_move':
            self.engine.move_drag(msg.get('dx', 0), msg.get('dy', 0))
            action_taken = True
            
        elif msg['eventType'] == 'drag_end':
            self.engine.end_drag()
            action_taken = True
            
        if action_taken:
            # Odświeżenie widoku przez Proxy
            self.proxy.img_data = self.engine.render()
            self.proxy._update_image(0, None)

# --- SETUP ---
engine = MyLargeObject()
proxy_w = SimpleRGBWidget(rgb_array=engine.render())
app = LabApp(engine, proxy_w)

meta = WindowMeta(width=512, height=512, offset_x=0, offset_y=0, name='Slice_Axial')
ui_lab = UICanvas(proxy_w, meta, app.handle_ui_event)
ui_lab.display()

In [ ]:
ui_canvas.msg.layout.height='400px'